# 🌌 Stellar Classification (SDSS) — GALAXY / QSO / STAR

**Goal:** classify sky objects into GALAXY, QSO, or STAR using photometric + spectral features.

**Final best test accuracy: 0.968** (macro F1 ≈ 0.96)

---
**My notes / thought process (POV):** Started with a basic LightGBM pipeline that got ~0.9618. From there I went through data profiling, feature engineering, feature pruning, class-imbalance checking, hyperparameter search (RandomizedSearchCV + Optuna), a leakage check, and a data-quality check on `redshift` — squeezed it up to 0.968. I've left my reasoning as markdown notes throughout so it's clear *why* each step was taken, not just *what* was done.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import lightgbm as lgb


## 2. Load data

In [ ]:
df = pd.read_csv('kaggle1.csv')
df.head()


## 3. Pandas Profiling — automated EDA

**POV:** Before touching any model, I ran a full profiling report on the raw dataframe. This is what first flagged `class` and `galaxy_population` as "High correlation" columns — which is exactly what led me to later double-check `galaxy_population` for target leakage (see Section 8). It also confirmed there are zero missing values anywhere, so no imputation is needed.

> Requires: `pip install ydata-profiling`

In [ ]:
# pip install ydata-profiling   # uncomment/run once if not already installed

from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Stellar Classification - Profiling Report", explorative=True)
profile.to_notebook_iframe()

# To save as a standalone HTML file instead of viewing inline:
# profile.to_file("pandas-profiling-df.html")


## 4. Encode target

`class` (GALAXY / QSO / STAR) is label-encoded to 0/1/2 so LightGBM can consume it. `le.classes_` remembers the mapping — used later with `target_names=le.classes_` in `classification_report` and with `le.inverse_transform()` whenever we need readable labels back.

In [ ]:
le = LabelEncoder()
df['class'] = le.fit_transform(df['class'])
print(dict(zip(le.classes_, le.transform(le.classes_))))


## 5. Class imbalance check

**POV:** Checked this because accuracy alone can hide a model that's just always guessing the majority class. Distribution turned out to be GALAXY 65.4% / QSO 20.3% / STAR 14.3% — moderate imbalance (~4.5:1 largest:smallest), not extreme. I tried `class_weight='balanced'` to compensate, but it didn't meaningfully change per-class recall, so it isn't in the final pipeline below — StratifiedKFold + StratifiedSplit were enough to keep things fair.

In [ ]:
print(df['class'].value_counts())
print()
print((df['class'].value_counts(normalize=True) * 100).round(2))


## 6. Feature engineering

**POV:** Photometric colors (differences between adjacent bands: u-g, g-r, r-i, i-z) are the astrophysically meaningful features here — they track redshift/temperature and are classic discriminators between stars, galaxies, and quasars. I also tried products (`u*g`, etc.) and sums (`u+g`, etc.) of bands; feature-importance analysis later (Section 11) showed colors and `ug`/`iz` pulled their weight, while `gr`, `ri`, and raw `r`/`i` bands were largely redundant — those got dropped in the pruned preprocessor (Section 9).

In [ ]:
# Color differences — the strongest engineered features
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']

# Products — kept ug & iz after pruning, dropped gr & ri (see Section 9/11)
df['ug'] = df['u'] * df['g']
df['gr'] = df['g'] * df['r']
df['ri'] = df['r'] * df['i']
df['iz'] = df['i'] * df['z']

df.head()


## 7. Feature/target split + train-test split

`stratify=y` keeps the GALAXY/QSO/STAR proportions identical in train and test — important given the imbalance noted above. This split is done **once**, up front, and `X_test`/`y_test` are never touched again until final evaluation (Section 10) — no leakage into training or CV.

In [ ]:
X = df.drop(columns=['class'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## 8. Leakage check — `galaxy_population`

**POV:** The profiling report (Section 3) flagged `galaxy_population` as "High correlation" with `class`, same as it flagged `class` itself — which made me suspicious it might be an indirect proxy for the target (i.e. leakage).

Crosstab result:

| galaxy_population | GALAXY | QSO | STAR |
|---|---|---|---|
| Blue_Cloud | 88,962 | 108,274 | 60,546 |
| Red_Sequence | 288,518 | 8,869 | 22,178 |

**Verdict: not leakage.** Both values appear across all three classes in meaningful numbers — if this were leakage, one value would map almost 1:1 to a single class. It's a real, if skewed, feature. (It's currently *not* included in the model below — that's a possible future improvement, not a decision I made deliberately either way.)

In [ ]:
print(pd.crosstab(df['galaxy_population'], le.inverse_transform(df['class'])))


## 9. Preprocessing (pruned feature set)

**POV:** `numerical_features_v2` below is the *second* version of my feature list — the first version used every raw band + every engineered feature, and scored ~0.9618. After looking at gain-based feature importance (Section 11), `r`, `i`, `gr`, and `ri` were consistently at the bottom, so I dropped them. That alone, combined with StratifiedKFold + a wider hyperparameter search, is most of what pushed accuracy from 0.9618 → 0.968.

**Also checked:** `redshift` has ~8,957 rows (1.55%) with tiny negative values (max magnitude only -0.00997) — almost certainly measurement noise for near-zero-redshift STAR/nearby-GALAXY objects, not genuine blueshift. I tried `df['redshift'].clip(lower=0)` to "clean" this — **accuracy went down slightly**, so I reverted it. Lesson: what looks like noise sometimes carries real signal for a tree model; raw values are kept here.

In [ ]:
numerical_features_v2 = [
    'alpha', 'delta', 'u', 'g', 'z', 'redshift',
    'u_g', 'g_r', 'r_i', 'i_z', 'ug', 'iz'
    # dropped as weak (low LightGBM gain importance): 'r', 'i', 'gr', 'ri'
]
categorical_features = ['spectral_type']

preprocessor_v2 = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features_v2),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


## 10. Pipeline + hyperparameter search

**POV:** I compared two search strategies here:
- **RandomizedSearchCV** (below) with a wide grid + `n_iter=50` + `StratifiedKFold(5)` → **0.968 test accuracy**
- **Optuna** (Bayesian optimization, ~50 trials, same CV) → came out slightly *lower* on this dataset/search-budget combination.

That's a bit counter-intuitive since Optuna is usually the smarter search, but with a similar trial budget, random search got lucky/thorough enough here. I went with whichever actually scored higher rather than assuming one method is always better — **RandomizedSearchCV is the one used for the final model.**

In [ ]:
clf = Pipeline(steps=[
    ("preprocessor", preprocessor_v2),
    ("model", lgb.LGBMClassifier(
        objective="multiclass",
        num_class=3,
        metric="multi_logloss",
        boosting_type="gbdt",
        random_state=42
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    "model__n_estimators": [100, 200, 300, 400, 500, 600, 800, 1000],
    "model__learning_rate": [0.005, 0.01, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
    "model__max_depth": [-1, 3, 4, 5, 6, 8, 10, 12],
    "model__num_leaves": [15, 20, 31, 40, 63, 80, 100, 127, 150],
    "model__min_child_samples": [5, 10, 20, 30, 50, 70, 100],
    "model__subsample": [0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10],
    "model__reg_lambda": [0, 0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10],
    "model__min_split_gain": [0, 0.01, 0.05, 0.1, 0.2],
}

random_search = RandomizedSearchCV(
    clf,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42,
    verbose=2
)


### Train

This is the slow step (50 iterations × 5 folds = 250 model fits on the full ~460k-row training set). Grab a coffee ☕.

In [ ]:
random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV f1_macro score:", random_search.best_score_)


## 11. Evaluate on the held-out test set

**POV:** This is the number that matters — `X_test` was never seen during training or cross-validation, so this is an honest, unleaked estimate of real-world performance.

I deliberately did **not** refit `best_estimator_` on the full `X, y` before this evaluation (that would leak `X_test` into training and inflate the score). `random_search.predict()` already uses `best_estimator_`, which sklearn's `RandomizedSearchCV(refit=True)` (the default) already refit on the full `X_train, y_train` internally — no extra `.fit()` call needed.

In [ ]:
y_pred = random_search.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


## 12. Confusion matrix

**POV:** This is where I found the most interesting pattern. Almost all of the model's errors are **STAR misclassified as GALAXY** (1,148 cases) and, to a lesser extent, **GALAXY as STAR** (881 cases). GALAXY↔QSO and QSO↔STAR confusion is much smaller.

I dug into *why*: pulled out the misclassified-STAR-as-GALAXY rows and compared their `redshift` distribution against correctly-classified STAR rows. The misclassified ones had roughly **2x higher median redshift** (~0.093 vs ~0.055) — i.e. these are STAR objects whose measured redshift looks anomalously galaxy-like, so the model (reasonably) leans GALAXY. This looks like either unusual stars or some label noise in the original data, not a flaw in the model itself.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


## 13. Feature importance

`importance_type='gain'` — how much each feature actually reduced loss when used in a split, which is more meaningful than raw split-count. `redshift` dominates (as expected astrophysically), followed by the `g_r` color — this is what originally justified pruning the weak raw bands/products in Section 9.

In [ ]:
lgb.plot_importance(
    random_search.best_estimator_.named_steps['model'],
    max_num_features=20,
    importance_type='gain'
)
plt.tight_layout()
plt.show()


## 14. Summary

- **Best test accuracy: 0.968** (macro F1 ≈ 0.96, weighted F1 ≈ 0.97)
- Per-class recall: GALAXY 0.98, QSO 0.96, STAR 0.92
- Went from 0.9618 → 0.968 via: feature pruning (importance-based), `StratifiedKFold`, a wider `RandomizedSearchCV` grid, and ruling out a couple of false leads (Optuna didn't beat RandomizedSearchCV here; clipping negative `redshift` hurt rather than helped)
- **Validated the result is trustworthy:** checked `galaxy_population` for target leakage (clean — genuine feature, not a proxy), and confirmed no missing values anywhere via the profiling report
- Remaining errors are concentrated in STAR↔GALAXY confusion, tied to a subset of STAR objects with anomalously galaxy-like redshift — likely close to the practical ceiling for this feature set without new data or domain features

**Possible next steps if pushing further:**
- Add `galaxy_population` as a second categorical feature (confirmed clean, currently unused)
- Stack LightGBM + XGBoost
- A redshift-based interaction feature or binning, specifically targeting the STAR↔GALAXY boundary
- Manually inspect the ~1,148 misclassified STAR rows for a data-quality/labeling issue